In [102]:
# Standard libraries
import os
from typing import List, Tuple

# Data manipulation and numerical computation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn utilities
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV,cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif, mutual_info_regression, f_classif, f_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from catboost import CatBoostRegressor

# Other utilities
from scipy.stats import randint, uniform
import joblib

# Standard libraries
from typing import List, Tuple

# Data manipulation and numerical computation
import numpy as np
import pandas as pd

# Scikit-learn utilities
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold, f_classif, f_regression

In [103]:
path = "../../data/processed/"
dfs_processed = {}

# read all dataframes and keep them in dfs
for parquet in os.listdir(path):  # List all files in the directory
    if parquet.endswith(".parquet"):
        name = parquet.split(".parquet")[0]  # Get name without extension
        dfs_processed[name] = pd.read_parquet(os.path.join(path, parquet))  # Use os.path.join for paths
        print(f"Loaded {name} with shape {dfs_processed[name].shape}")

Loaded 03_CLEAN_COMPLETE_DF with shape (49863, 174)
Loaded 03_CLEAN_COMPLETE_DF_02 with shape (49231, 623)
Loaded 03_COMPLETE_TEST with shape (43568, 3)
Loaded 03_COMPLETE_TRAIN with shape (49441, 2332)
Loaded 03_COMPLETE_TRAIN_2 with shape (49231, 2849)
Loaded clean_train with shape (43568, 4)
Loaded dep_codes with shape (49231, 12)
Loaded dep_test with shape (5063, 12)
Loaded taxones_pressure with shape (5663, 2333)
Loaded taxones_pressure_epm_predict with shape (5663, 2850)
Loaded taxones_pressure_epm_train with shape (43568, 2853)
Loaded taxones_pressure_predict with shape (5663, 2333)
Loaded taxones_pressure_train with shape (43568, 2336)


In [104]:
sites = dfs_processed[list(dfs_processed.keys())[6]]

In [105]:
regiones = sites['HERlvl1Code'].drop_duplicates().tolist()
len(regiones)

22

In [106]:
path = "../../notebooks/06_cb_regression/dfs_taxon_p/"
dfs = {}

# read all dataframes and keep them in dfs
for parquet in os.listdir(path):  # List all files in the directory
    if parquet.endswith(".parquet"):
        name = parquet.split(".parquet")[0]  # Get name without extension
        dfs[name] = pd.read_parquet(os.path.join(path, parquet))  # Use os.path.join for paths
        print(f"Loaded {name} with shape {dfs[name].shape}")

Loaded df_1 with shape (1485, 109)
Loaded df_10 with shape (3926, 148)
Loaded df_11 with shape (1289, 163)
Loaded df_12 with shape (4677, 176)
Loaded df_13 with shape (1003, 172)
Loaded df_14 with shape (7742, 161)
Loaded df_15 with shape (1223, 160)
Loaded df_16 with shape (379, 113)
Loaded df_17 with shape (803, 151)
Loaded df_18 with shape (1164, 152)
Loaded df_19 with shape (500, 134)
Loaded df_2 with shape (352, 97)
Loaded df_20 with shape (508, 209)
Loaded df_21 with shape (2663, 184)
Loaded df_22 with shape (152, 175)
Loaded df_3 with shape (4996, 141)
Loaded df_4 with shape (596, 141)
Loaded df_5 with shape (2313, 131)
Loaded df_6 with shape (2134, 142)
Loaded df_7 with shape (524, 107)
Loaded df_8 with shape (548, 123)
Loaded df_9 with shape (10254, 169)


In [107]:
from types import SimpleNamespace

# Después de llenar `dfs`:
d = SimpleNamespace(**dfs)

# Usos:
d.df_1.head()
d.df_10.shape

(3926, 148)

In [108]:
cleandf = d.df_1

In [109]:
def crear_modelo(cleandf):
    cleandf = cleandf[cleandf['IBD'].notna()]
    X = cleandf.drop(columns=['IBD','IBD_EQR','IBD_EQR_Status'])
    y = cleandf['IBD']
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
    # 2) Identify column types
    num_cols = X.select_dtypes(include=['number']).columns
    cat_cols = X.columns.difference(num_cols)
    # 3) Preprocess
    # this preprocessor can handle missing values   
    pre = ColumnTransformer([
        # numerical features
        ('num', Pipeline([
            # imputation and scaling
            ('imp', SimpleImputer(strategy='median')),
            # scaling (RF doesn't need it, but other models might)
            # ('scaler', StandardScaler(with_mean=False))  # stays sparse with OHE
        ]), num_cols),
        # categorical features
        ('cat', Pipeline([
            # imputation as None being another category
            ('imp', SimpleImputer(strategy='constant', fill_value='None')),
            # one-hot encoding, ignoring unknown categories during inference
            ('ohe', OneHotEncoder(handle_unknown='ignore'))
        ]), cat_cols)

    ])
    # 4) Model
    clf = Pipeline([
        ('pre', pre),
        ('model', RandomForestRegressor(n_estimators=600, random_state=42, ))  # for regression
    ])  
    clf.fit(X_tr, y_tr)
    print("R2 train:", clf.score(X_tr, y_tr))
    print("R2 valid:", clf.score(X_te, y_te))

    return clf

In [110]:
clf = crear_modelo(cleandf)

R2 train: 0.9571326307216331
R2 valid: 0.7015659552614844


In [111]:

def train_region_model(cleandf, target='IBD'):
    # 1) separar train y scoring dentro de la MISMA región
    df_train = cleandf[cleandf[target].notna()].copy()
    df_score = cleandf[cleandf[target].isna()].copy()

    # X / y
    drop_cols = [c for c in ['IBD','IBD_EQR','IBD_EQR_Status'] if c in cleandf.columns]
    X = df_train.drop(columns=drop_cols)
    y = df_train[target].astype(float)

    # 2) columnas num/cat
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

    pre = ColumnTransformer([
        ('num', Pipeline([
            ('imp', SimpleImputer(strategy='median')),
        ]), num_cols),
        ('cat', Pipeline([
            ('imp', SimpleImputer(strategy='constant', fill_value='(missing)')),
            ('ohe', OneHotEncoder(handle_unknown='ignore')),
        ]), cat_cols)
    ])

    # 3) modelo
    clf = Pipeline([
        ('pre', pre),
        ('model', RandomForestRegressor(n_estimators=600, 
                                        random_state=42, 
                                        n_jobs=-1))
    ])

    # 4) hold-out (barajado dentro de la región)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

    clf.fit(X_tr, y_tr)
    pred_tr = clf.predict(X_tr)
    pred_te = clf.predict(X_te)

    metrics = {
        'R2_train': r2_score(y_tr, pred_tr),
        'R2_valid': r2_score(y_te, pred_te),
        'MAE_train': mean_absolute_error(y_tr, pred_tr),
        'MAE_valid': mean_absolute_error(y_te, pred_te),
        'RMSE_train': mean_squared_error(y_tr, pred_tr),
        'RMSE_valid': mean_squared_error(y_te, pred_te),
    }

    # 5) (opcional) KFold simple dentro de la región
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_r2 = cross_val_score(clf, X, y, cv=kf, scoring='r2', n_jobs=-1)
    metrics['R2_CV_mean'] = cv_r2.mean()
    metrics['R2_CV_all']  = cv_r2

    # 6) predecir filas sin target de esta región
    if not df_score.empty:
        X_score = df_score.drop(columns=drop_cols, errors='ignore')
        preds_score = clf.predict(X_score)
        scored = df_score.copy()
        scored[target + '_pred'] = preds_score
    else:
        scored = pd.DataFrame(columns=cleandf.columns.tolist() + [target + '_pred'])

    return clf, metrics, scored


In [112]:
model, metrics, scored = train_region_model(cleandf, target='IBD')
print(metrics)
# scored trae las filas SIN IBD con la columna IBD_pred

{'R2_train': 0.9573146344496368, 'R2_valid': 0.7030572518961957, 'MAE_train': 0.1369796825396837, 'MAE_valid': 0.39357477820025566, 'RMSE_train': 0.08077274328042393, 'RMSE_valid': 0.6540612109210022, 'R2_CV_mean': np.float64(0.7083962055295936), 'R2_CV_all': array([0.69812618, 0.77520163, 0.69350303, 0.74466357, 0.63048662])}


In [113]:

def train_catboost_region(
    cleandf: pd.DataFrame,
    target: str = 'IBD',
    test_size: float = 0.20,
    random_state: int = 42,
    early_stopping_rounds: int = 200,
    cat_params: dict | None = None,
):
    """
    Entrena CatBoost para una región usando cleandf.
    - Separa train (con target) y score (sin target)
    - Preprocesa (imputación num/cat + OHE)
    - Entrena con early stopping
    - Regresa: modelo (Pipeline), métricas, scored_df (filas sin target con predicción)
    """
    # 1) separar train / score
    df_train = cleandf[cleandf[target].notna()].copy()
    df_score = cleandf[cleandf[target].isna()].copy()

    # X / y
    drop_cols = [c for c in ['IBD','IBD_EQR','IBD_EQR_Status'] if c in cleandf.columns]
    X = df_train.drop(columns=drop_cols, errors='ignore')
    y = df_train[target].astype(float)

    # split
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=test_size, random_state=random_state)

    # 2) columnas num/cat
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

    # preprocesamiento
    pre = ColumnTransformer([
        ('num', Pipeline([
            ('imp', SimpleImputer(strategy='median')),
        ]), num_cols),
        ('cat', Pipeline([
            ('imp', SimpleImputer(strategy='constant', fill_value='(missing)')),
            ('ohe', OneHotEncoder(handle_unknown='ignore')),
        ]), cat_cols)
    ])

    # 3) modelo CatBoost (parámetros por defecto + override opcional)
    base_params = dict(
        depth=6, learning_rate=0.05, n_estimators=3000,
        loss_function='RMSE', random_state=random_state, verbose=0
    )
    if cat_params:
        base_params.update(cat_params)

    cb = Pipeline([
        ('pre', pre),
        ('model', CatBoostRegressor(**base_params))
    ])

    # 4) ajustar: primero ajustamos el preprocesador para armar matrices, luego el modelo con early stopping
    Xtr_proc = pre.fit_transform(X_tr)
    Xte_proc = pre.transform(X_te)
    cb.named_steps['model'].fit(
        Xtr_proc, y_tr,
        eval_set=(Xte_proc, y_te),
        use_best_model=True,
        early_stopping_rounds=early_stopping_rounds
    )

    # 5) métricas en hold-out y train (usando el pipeline para que transforme igual)
    r2_tr  = cb.score(X_tr, y_tr)
    r2_te  = cb.score(X_te, y_te)
    pred_tr = cb.predict(X_tr); pred_te = cb.predict(X_te)
    metrics = {
        'R2_train': r2_tr,
        'R2_valid': r2_te,
        'MAE_train': mean_absolute_error(y_tr, pred_tr),
        'MAE_valid': mean_absolute_error(y_te, pred_te),
        'RMSE_train': mean_squared_error(y_tr, pred_tr),
        'RMSE_valid': mean_squared_error(y_te, pred_te),
        'best_iterations': int(cb.named_steps['model'].get_best_iteration() or base_params['n_estimators'])
    }

    # 6) predicciones para filas sin target de esta región (si existen)
    if not df_score.empty:
        X_score = df_score.drop(columns=drop_cols, errors='ignore')
        df_score[target + '_pred'] = cb.predict(X_score)
        scored_df = df_score
        
    else:
        scored_df = pd.DataFrame(columns=list(cleandf.columns) + [target + '_pred'])

    return cb, metrics, scored_df

In [114]:
model, metrics, scored = train_catboost_region(cleandf, target='IBD')
print(metrics)
# 'scored' contiene las filas sin IBD con la columna IBD_pred

{'R2_train': np.float64(0.9997315978106086), 'R2_valid': np.float64(0.7719420882757426), 'MAE_train': 0.019290269418419926, 'MAE_valid': 0.3375528071183224, 'RMSE_train': 0.000507892596445965, 'RMSE_valid': 0.5023319641749209, 'best_iterations': 2999}


In [115]:
region=1
attr = f"df_{int(region)}"           # df_1, df_2, df_18, ...
if not hasattr(d, attr):
    # fallback por si tus nombres vienen con ceros: df_01
    attr = f"df_{str(region)}"
if not hasattr(d, attr):
    raise AttributeError(f"No encontré {attr} en d")

cleandf = getattr(d, attr) 
model, m, scored = train_catboost_region(cleandf, target='IBD') 

In [116]:
predicciones = []
all_metrics = {}  # guardará {region: métricas}

for region in regiones:
    attr = f"df_{int(region)}"           # df_1, df_2, df_18, ...
    if not hasattr(d, attr):
        # fallback por si tus nombres vienen con ceros: df_01
        attr = f"df_{str(region)}"
    if not hasattr(d, attr):
        raise AttributeError(f"No encontré {attr} en d")

    cleandf = getattr(d, attr)
    model, m, scored = train_catboost_region(cleandf, target='IBD')  # m = métricas de esa región
    all_metrics[region] = m

    # Mantener índice (SamplingOperations_code) y solo la predicción
    solo_ibd = scored[['IBD_pred']].copy()
    solo_ibd['region'] = region
    predicciones.append(solo_ibd)

predicciones = pd.concat(predicciones, axis=0)  # índice preservado
predicciones.index.name = 'SamplingOperations_code'

In [117]:
predicciones

,IBD_pred,region
SamplingOperations_code,,
977,14.281455,18
978,15.036842,18
979,13.809988,18
980,14.920675,18
981,15.891145,18
...,...,...
347,19.870728,2
348,19.348878,2
349,17.755988,2


In [118]:
all_metrics

{18: {'R2_train': np.float64(0.999983711655112),
  'R2_valid': np.float64(0.9017637397395603),
  'MAE_train': 0.007065205088705387,
  'MAE_valid': 0.47587652373938083,
  'RMSE_train': 6.880692815046856e-05,
  'RMSE_valid': 0.5397919609553081,
  'best_iterations': 2991},
 5: {'R2_train': np.float64(0.9996173724474062),
  'R2_valid': np.float64(0.9296569807490126),
  'MAE_train': 0.040646382907688024,
  'MAE_valid': 0.4253394641109129,
  'RMSE_train': 0.0023713147980632225,
  'RMSE_valid': 0.5048798877089307,
  'best_iterations': 2997},
 4: {'R2_train': np.float64(0.9999981945781938),
  'R2_valid': np.float64(0.8332813178627969),
  'MAE_train': 0.0030720960306941506,
  'MAE_valid': 0.7435964879526081,
  'RMSE_train': 1.3077189985678924e-05,
  'RMSE_valid': 1.2830969840288327,
  'best_iterations': 2196},
 10: {'R2_train': np.float64(0.9988717494434298),
  'R2_valid': np.float64(0.9497038321063085),
  'MAE_train': 0.07942961232981795,
  'MAE_valid': 0.3810493640175195,
  'RMSE_train': 0.00

In [119]:
metrics_df = (pd.DataFrame.from_dict(all_metrics, orient='index')
                .reset_index()
                .rename(columns={'index':'region'}))



In [120]:
metrics_df

,region,R2_train,R2_valid,MAE_train,MAE_valid,RMSE_train,RMSE_valid,best_iterations
0,18,0.999984,0.901764,0.007065,0.475877,6.880693e-05,0.539792,2991
1,5,0.999617,0.929657,0.040646,0.425339,2.371315e-03,0.504880,2997
2,4,0.999998,0.833281,0.003072,0.743596,1.307719e-05,1.283097,2196
3,10,0.998872,0.949704,0.079430,0.381049,9.270532e-03,0.404585,2993
4,22,1.000000,0.562962,0.000040,1.190931,1.913874e-09,3.673909,1652
5,9,0.989169,0.941368,0.145762,0.265904,3.568258e-02,0.189518,2985
6,21,0.999565,0.928841,0.045767,0.463063,3.003642e-03,0.525956,2999
7,20,1.000000,0.867077,0.000260,0.726611,8.974746e-08,0.842552,2980
8,12,0.997816,0.944985,0.088146,0.353910,1.179671e-02,0.311098,2995
9,8,0.999931,0.842503,0.014798,0.613522,2.899673e-04,0.686089,1482


In [121]:
metrics_df['gap']=metrics_df['R2_train']-metrics_df['R2_valid']
metrics_df

,region,R2_train,R2_valid,MAE_train,MAE_valid,RMSE_train,RMSE_valid,best_iterations,gap
0,18,0.999984,0.901764,0.007065,0.475877,6.880693e-05,0.539792,2991,0.098220
1,5,0.999617,0.929657,0.040646,0.425339,2.371315e-03,0.504880,2997,0.069960
2,4,0.999998,0.833281,0.003072,0.743596,1.307719e-05,1.283097,2196,0.166717
3,10,0.998872,0.949704,0.079430,0.381049,9.270532e-03,0.404585,2993,0.049168
4,22,1.000000,0.562962,0.000040,1.190931,1.913874e-09,3.673909,1652,0.437038
5,9,0.989169,0.941368,0.145762,0.265904,3.568258e-02,0.189518,2985,0.047801
6,21,0.999565,0.928841,0.045767,0.463063,3.003642e-03,0.525956,2999,0.070724
7,20,1.000000,0.867077,0.000260,0.726611,8.974746e-08,0.842552,2980,0.132923
8,12,0.997816,0.944985,0.088146,0.353910,1.179671e-02,0.311098,2995,0.052831
9,8,0.999931,0.842503,0.014798,0.613522,2.899673e-04,0.686089,1482,0.157427


In [122]:
# tabla de regiones: columnas -> HERlvl1Code, HERlvl1Name
reg_tab = sites[['HERlvl1Code','HERlvl1Name']].drop_duplicates()

# aseguramos tipos
predicciones['region'] = predicciones['region'].astype(int)
reg_tab['HERlvl1Code'] = reg_tab['HERlvl1Code'].astype(int)

# diccionario y nueva columna
name_map = dict(zip(reg_tab['HERlvl1Code'], reg_tab['HERlvl1Name']))
predicciones['HERlvl1Name'] = predicciones['region'].map(name_map)
predicciones.drop(columns=['region'])


,IBD_pred,HERlvl1Name
SamplingOperations_code,,
977,14.281455,ALSACE
978,15.036842,ALSACE
979,13.809988,ALSACE
980,14.920675,ALSACE
981,15.891145,ALSACE
...,...,...
347,19.870728,ALPES INTERNES
348,19.348878,ALPES INTERNES
349,17.755988,ALPES INTERNES


In [123]:
# load ranges
    # save to csv
ranges = pd.read_csv("../../data/processed/ibd_eqr_ranges_by_herlvl1_continuous_midpoint.csv")
ranges.head(20)

,HERlvl1Name,IBD_EQR_Status,IBD_min,IBD_max,IBD_mid
0,ALPES INTERNES,Bad,0.000,9.800,9.30
1,ALPES INTERNES,Poor,9.800,13.225,10.30
2,ALPES INTERNES,Moderate,13.225,17.025,16.15
3,ALPES INTERNES,Good,17.025,18.725,17.90
4,ALPES INTERNES,High,18.725,20.000,19.55
5,ALSACE,Bad,0.000,6.925,5.40
6,ALSACE,Poor,6.925,10.425,8.45
7,ALSACE,Moderate,10.425,14.050,12.40
8,ALSACE,Good,14.050,17.125,15.70
9,ALSACE,High,17.125,20.000,18.55


In [124]:
def add_eqr_status(yhat: pd.DataFrame, ranges: pd.DataFrame) -> pd.DataFrame:
    """
    Adds the column 'IBD_EQR_Status_Predicted' to the `yhat` DataFrame by mapping the predicted IBD values 
    ('IBD_Predicted') into the appropriate bin defined by the [IBD_min, IBD_max) intervals in the `ranges` DataFrame 
    for the corresponding 'HERlvl1Name'. The topmost bin per region also includes its right endpoint.

    Parameters:
    ----------
    yhat : pd.DataFrame
        A DataFrame containing the predicted IBD values ('IBD_Predicted') and the corresponding 'HERlvl1Name'.
    ranges : pd.DataFrame
        A DataFrame containing the bin definitions for each 'HERlvl1Name', including columns:
        - 'HERlvl1Name': The region name.
        - 'IBD_EQR_Status': The status corresponding to the bin.
        - 'IBD_min': The lower bound of the bin (inclusive).
        - 'IBD_max': The upper bound of the bin (exclusive, except for the topmost bin).

    Returns:
    -------
    pd.DataFrame
        A copy of the `yhat` DataFrame with an additional column 'IBD_EQR_Status_Predicted', which contains the 
        mapped status for each prediction.

    Notes:
    -----
    - The function performs a cartesian merge between `yhat` and `ranges` based on 'HERlvl1Name'.
    - Each predicted value is matched to the bin where it falls within the [IBD_min, IBD_max) interval.
    - For the topmost bin in each region, the right endpoint (IBD_max) is included.
    - In case of ties (multiple bins matching a prediction), the first match is kept.
    """
    out = yhat.copy()
    out['__ix__'] = np.arange(len(out))

    # Copy ranges and calculate the maximum right endpoint for each region
    r = ranges[['HERlvl1Name', 'IBD_EQR_Status', 'IBD_min', 'IBD_max']].copy()
    r['__max_right__'] = r.groupby('HERlvl1Name')['IBD_max'].transform('max')

    # Cartesian merge by region, then keep the single interval that matches each prediction
    m = out.merge(r, on='HERlvl1Name', how='left')

    # Check if predictions fall within the bin intervals
    pred = m['IBD_pred'].astype(float)
    left_ok  = pred >= m['IBD_min']
    right_ok = (pred <  m['IBD_max']) | ((pred == m['IBD_max']) & (m['IBD_max'].eq(m['__max_right__'])))
    m = m[left_ok & right_ok]

    # In case of any ties, keep the first match; then map back to original rows
    m = m.sort_values(['__ix__', 'IBD_min', 'IBD_max']).drop_duplicates('__ix__', keep='first')
    status = m.set_index('__ix__')['IBD_EQR_Status']

    # Map the status back to the original DataFrame
    out['IBD_EQR_Status_Predicted'] = out['__ix__'].map(status)
    out = out.drop(columns='__ix__')
    return out

In [125]:
yhat2 = add_eqr_status(predicciones, ranges)
yhat2

,IBD_pred,region,HERlvl1Name,IBD_EQR_Status_Predicted
SamplingOperations_code,,,,
977,14.281455,18,ALSACE,Good
978,15.036842,18,ALSACE,Good
979,13.809988,18,ALSACE,Moderate
980,14.920675,18,ALSACE,Good
981,15.891145,18,ALSACE,Good
...,...,...,...,...
347,19.870728,2,ALPES INTERNES,High
348,19.348878,2,ALPES INTERNES,High
349,17.755988,2,ALPES INTERNES,Good


In [126]:
yhat2_clean = yhat2[yhat2['IBD_EQR_Status_Predicted'].notna()].copy()
len(yhat2_clean)

5415

In [127]:
dirty = yhat2[yhat2['IBD_EQR_Status_Predicted'].isna()].copy()
dirty

,IBD_pred,region,HERlvl1Name,IBD_EQR_Status_Predicted
SamplingOperations_code,,,,
1077,20.205546,18,ALSACE,NaN
2072,20.179152,5,JURA-PREALPES DU NORD,NaN
2073,20.050622,5,JURA-PREALPES DU NORD,NaN
2074,20.022628,5,JURA-PREALPES DU NORD,NaN
2090,20.048645,5,JURA-PREALPES DU NORD,NaN
...,...,...,...,...
334,20.011994,2,ALPES INTERNES,NaN
336,20.022435,2,ALPES INTERNES,NaN
341,20.001372,2,ALPES INTERNES,NaN


In [128]:
print(np.min(dirty['IBD_pred']))
print(np.max(dirty['IBD_pred']))

20.000038221584315
21.571141312337826


In [129]:
yhat2['IBD_EQR_Status_Predicted'] = yhat2['IBD_EQR_Status_Predicted'].fillna('High')

In [130]:
send_predictions = yhat2['IBD_EQR_Status_Predicted']

In [131]:
send_predictions

SamplingOperations_code
977        Good
978        Good
979    Moderate
980        Good
981        Good
         ...   
347        High
348        High
349        Good
350        High
351        High
Name: IBD_EQR_Status_Predicted, Length: 5663, dtype: object

In [132]:
# change the name to "IBD_EQR_Status"
truesend = send_predictions.rename("IBD_EQR_Status")
#truesend.to_csv("cat_boost_perh.csv")
truesend

SamplingOperations_code
977        Good
978        Good
979    Moderate
980        Good
981        Good
         ...   
347        High
348        High
349        Good
350        High
351        High
Name: IBD_EQR_Status, Length: 5663, dtype: object

In [133]:
path = "../../notebooks/06_cb_regression/dfs_taxon_p_epm/"
dfs = {}

# read all dataframes and keep them in dfs
for parquet in os.listdir(path):  # List all files in the directory
    if parquet.endswith(".parquet"):
        name = parquet.split(".parquet")[0]  # Get name without extension
        dfs[name] = pd.read_parquet(os.path.join(path, parquet))  # Use os.path.join for paths
        print(f"Loaded {name} with shape {dfs[name].shape}")

Loaded df_1 with shape (1485, 148)
Loaded df_10 with shape (3926, 504)
Loaded df_11 with shape (1289, 281)
Loaded df_12 with shape (4677, 434)
Loaded df_13 with shape (1003, 269)
Loaded df_14 with shape (7742, 329)
Loaded df_15 with shape (1223, 434)
Loaded df_16 with shape (379, 336)
Loaded df_17 with shape (803, 246)
Loaded df_18 with shape (1164, 447)
Loaded df_19 with shape (500, 220)
Loaded df_2 with shape (352, 360)
Loaded df_20 with shape (508, 364)
Loaded df_21 with shape (2663, 298)
Loaded df_22 with shape (152, 434)
Loaded df_3 with shape (4996, 217)
Loaded df_4 with shape (596, 442)
Loaded df_5 with shape (2313, 411)
Loaded df_6 with shape (2134, 423)
Loaded df_7 with shape (524, 352)
Loaded df_8 with shape (548, 232)
Loaded df_9 with shape (10254, 516)


In [134]:
predicciones_epm = []
all_metrics_epm = {}  # guardará {region: métricas}

for region in regiones:
    attr = f"df_{int(region)}"           # df_1, df_2, df_18, ...
    if not hasattr(d, attr):
        # fallback por si tus nombres vienen con ceros: df_01
        attr = f"df_{str(region)}"
    if not hasattr(d, attr):
        raise AttributeError(f"No encontré {attr} en d")

    cleandf = getattr(d, attr)
    model, m, scored = train_catboost_region(cleandf, target='IBD')  # m = métricas de esa región
    all_metrics_epm[region] = m

    # Mantener índice (SamplingOperations_code) y solo la predicción
    solo_ibd = scored[['IBD_pred']].copy()
    solo_ibd['region'] = region
    predicciones_epm.append(solo_ibd)

predicciones_epm = pd.concat(predicciones_epm, axis=0)  # índice preservado
predicciones_epm.index.name = 'SamplingOperations_code'

In [135]:
predicciones_epm

,IBD_pred,region
SamplingOperations_code,,
977,14.281455,18
978,15.036842,18
979,13.809988,18
980,14.920675,18
981,15.891145,18
...,...,...
347,19.870728,2
348,19.348878,2
349,17.755988,2


In [136]:
metrics_epm_df = (pd.DataFrame.from_dict(all_metrics_epm, orient='index')
                .reset_index()
                .rename(columns={'index':'region'}))

metrics_epm_df['gap']=metrics_epm_df['R2_train']-metrics_epm_df['R2_valid']

metrics_epm_df.columns = [
    col if col == 'region' else col + '_epm'
    for col in metrics_epm_df.columns
]

metrics_epm_df

,region,R2_train_epm,R2_valid_epm,MAE_train_epm,MAE_valid_epm,RMSE_train_epm,RMSE_valid_epm,best_iterations_epm,gap_epm
0,18,0.999984,0.901764,0.007065,0.475877,6.880693e-05,0.539792,2991,0.098220
1,5,0.999617,0.929657,0.040646,0.425339,2.371315e-03,0.504880,2997,0.069960
2,4,0.999998,0.833281,0.003072,0.743596,1.307719e-05,1.283097,2196,0.166717
3,10,0.998872,0.949704,0.079430,0.381049,9.270532e-03,0.404585,2993,0.049168
4,22,1.000000,0.562962,0.000040,1.190931,1.913874e-09,3.673909,1652,0.437038
5,9,0.989169,0.941368,0.145762,0.265904,3.568258e-02,0.189518,2985,0.047801
6,21,0.999565,0.928841,0.045767,0.463063,3.003642e-03,0.525956,2999,0.070724
7,20,1.000000,0.867077,0.000260,0.726611,8.974746e-08,0.842552,2980,0.132923
8,12,0.997816,0.944985,0.088146,0.353910,1.179671e-02,0.311098,2995,0.052831
9,8,0.999931,0.842503,0.014798,0.613522,2.899673e-04,0.686089,1482,0.157427


In [137]:
path = "../../notebooks/06_cb_regression/dfs_taxon_p_epm_dirty_af/"
dfs = {}

# read all dataframes and keep them in dfs
for parquet in os.listdir(path):  # List all files in the directory
    if parquet.endswith(".parquet"):
        name = parquet.split(".parquet")[0]  # Get name without extension
        dfs[name] = pd.read_parquet(os.path.join(path, parquet))  # Use os.path.join for paths
        print(f"Loaded {name} with shape {dfs[name].shape}")

Loaded df_1 with shape (1485, 322)
Loaded df_10 with shape (3926, 597)
Loaded df_11 with shape (1289, 405)
Loaded df_12 with shape (4677, 569)
Loaded df_13 with shape (1003, 430)
Loaded df_14 with shape (7742, 475)
Loaded df_15 with shape (1223, 527)
Loaded df_16 with shape (379, 418)
Loaded df_17 with shape (803, 372)
Loaded df_18 with shape (1164, 530)
Loaded df_19 with shape (500, 360)
Loaded df_2 with shape (352, 423)
Loaded df_20 with shape (508, 551)
Loaded df_21 with shape (2663, 530)
Loaded df_22 with shape (152, 559)
Loaded df_3 with shape (4996, 421)
Loaded df_4 with shape (596, 511)
Loaded df_5 with shape (2313, 501)
Loaded df_6 with shape (2134, 517)
Loaded df_7 with shape (524, 431)
Loaded df_8 with shape (548, 415)
Loaded df_9 with shape (10254, 615)


In [138]:
predicciones_dirty = []
all_metrics_dirty = {}  # guardará {region: métricas}

for region in regiones:
    attr = f"df_{int(region)}"           # df_1, df_2, df_18, ...
    if not hasattr(d, attr):
        # fallback por si tus nombres vienen con ceros: df_01
        attr = f"df_{str(region)}"
    if not hasattr(d, attr):
        raise AttributeError(f"No encontré {attr} en d")

    cleandf = getattr(d, attr)
    model, m, scored = train_catboost_region(cleandf, target='IBD')  # m = métricas de esa región
    all_metrics_dirty[region] = m

    # Mantener índice (SamplingOperations_code) y solo la predicción
    solo_ibd = scored[['IBD_pred']].copy()
    solo_ibd['region'] = region
    predicciones_dirty.append(solo_ibd)

predicciones_dirty = pd.concat(predicciones_dirty, axis=0)  # índice preservado
predicciones_dirty.index.name = 'SamplingOperations_code'

In [139]:
predicciones_dirty

,IBD_pred,region
SamplingOperations_code,,
977,14.281455,18
978,15.036842,18
979,13.809988,18
980,14.920675,18
981,15.891145,18
...,...,...
347,19.870728,2
348,19.348878,2
349,17.755988,2


In [140]:
metrics_dirty_df = (pd.DataFrame.from_dict(all_metrics_dirty, orient='index')
                .reset_index()
                .rename(columns={'index':'region'}))

metrics_dirty_df['gap']=metrics_dirty_df['R2_train']-metrics_dirty_df['R2_valid']

metrics_dirty_df.columns = [
    col if col == 'region' else col + '_dirty'
    for col in metrics_dirty_df.columns
]

metrics_dirty_df

,region,R2_train_dirty,R2_valid_dirty,MAE_train_dirty,MAE_valid_dirty,RMSE_train_dirty,RMSE_valid_dirty,best_iterations_dirty,gap_dirty
0,18,0.999984,0.901764,0.007065,0.475877,6.880693e-05,0.539792,2991,0.098220
1,5,0.999617,0.929657,0.040646,0.425339,2.371315e-03,0.504880,2997,0.069960
2,4,0.999998,0.833281,0.003072,0.743596,1.307719e-05,1.283097,2196,0.166717
3,10,0.998872,0.949704,0.079430,0.381049,9.270532e-03,0.404585,2993,0.049168
4,22,1.000000,0.562962,0.000040,1.190931,1.913874e-09,3.673909,1652,0.437038
5,9,0.989169,0.941368,0.145762,0.265904,3.568258e-02,0.189518,2985,0.047801
6,21,0.999565,0.928841,0.045767,0.463063,3.003642e-03,0.525956,2999,0.070724
7,20,1.000000,0.867077,0.000260,0.726611,8.974746e-08,0.842552,2980,0.132923
8,12,0.997816,0.944985,0.088146,0.353910,1.179671e-02,0.311098,2995,0.052831
9,8,0.999931,0.842503,0.014798,0.613522,2.899673e-04,0.686089,1482,0.157427


In [141]:
comparacion = pd.merge(metrics_df,metrics_epm_df,on='region',how='inner')
comparacion = pd.merge(comparacion, metrics_dirty_df, on='region', how='inner')

In [142]:
pd.set_option('display.max_columns', None)

In [143]:
comparacion

,region,R2_train,R2_valid,MAE_train,MAE_valid,RMSE_train,RMSE_valid,best_iterations,gap,R2_train_epm,R2_valid_epm,MAE_train_epm,MAE_valid_epm,RMSE_train_epm,RMSE_valid_epm,best_iterations_epm,gap_epm,R2_train_dirty,R2_valid_dirty,MAE_train_dirty,MAE_valid_dirty,RMSE_train_dirty,RMSE_valid_dirty,best_iterations_dirty,gap_dirty
0,18,0.999984,0.901764,0.007065,0.475877,6.880693e-05,0.539792,2991,0.098220,0.999984,0.901764,0.007065,0.475877,6.880693e-05,0.539792,2991,0.098220,0.999984,0.901764,0.007065,0.475877,6.880693e-05,0.539792,2991,0.098220
1,5,0.999617,0.929657,0.040646,0.425339,2.371315e-03,0.504880,2997,0.069960,0.999617,0.929657,0.040646,0.425339,2.371315e-03,0.504880,2997,0.069960,0.999617,0.929657,0.040646,0.425339,2.371315e-03,0.504880,2997,0.069960
2,4,0.999998,0.833281,0.003072,0.743596,1.307719e-05,1.283097,2196,0.166717,0.999998,0.833281,0.003072,0.743596,1.307719e-05,1.283097,2196,0.166717,0.999998,0.833281,0.003072,0.743596,1.307719e-05,1.283097,2196,0.166717
3,10,0.998872,0.949704,0.079430,0.381049,9.270532e-03,0.404585,2993,0.049168,0.998872,0.949704,0.079430,0.381049,9.270532e-03,0.404585,2993,0.049168,0.998872,0.949704,0.079430,0.381049,9.270532e-03,0.404585,2993,0.049168
4,22,1.000000,0.562962,0.000040,1.190931,1.913874e-09,3.673909,1652,0.437038,1.000000,0.562962,0.000040,1.190931,1.913874e-09,3.673909,1652,0.437038,1.000000,0.562962,0.000040,1.190931,1.913874e-09,3.673909,1652,0.437038
5,9,0.989169,0.941368,0.145762,0.265904,3.568258e-02,0.189518,2985,0.047801,0.989169,0.941368,0.145762,0.265904,3.568258e-02,0.189518,2985,0.047801,0.989169,0.941368,0.145762,0.265904,3.568258e-02,0.189518,2985,0.047801
6,21,0.999565,0.928841,0.045767,0.463063,3.003642e-03,0.525956,2999,0.070724,0.999565,0.928841,0.045767,0.463063,3.003642e-03,0.525956,2999,0.070724,0.999565,0.928841,0.045767,0.463063,3.003642e-03,0.525956,2999,0.070724
7,20,1.000000,0.867077,0.000260,0.726611,8.974746e-08,0.842552,2980,0.132923,1.000000,0.867077,0.000260,0.726611,8.974746e-08,0.842552,2980,0.132923,1.000000,0.867077,0.000260,0.726611,8.974746e-08,0.842552,2980,0.132923
8,12,0.997816,0.944985,0.088146,0.353910,1.179671e-02,0.311098,2995,0.052831,0.997816,0.944985,0.088146,0.353910,1.179671e-02,0.311098,2995,0.052831,0.997816,0.944985,0.088146,0.353910,1.179671e-02,0.311098,2995,0.052831
9,8,0.999931,0.842503,0.014798,0.613522,2.899673e-04,0.686089,1482,0.157427,0.999931,0.842503,0.014798,0.613522,2.899673e-04,0.686089,1482,0.157427,0.999931,0.842503,0.014798,0.613522,2.899673e-04,0.686089,1482,0.157427


In [144]:
# Diccionario para mapear nombres de columnas al modelo
modelo_map = {
    'gap': 'tp',
    'gap_epm': 'epm',
    'gap_dirty': 'dirty',
    'R2_train': 'tp',
    'R2_train_epm': 'epm',
    'R2_train_dirty': 'dirty',
    'R2_valid': 'tp',
    'R2_valid_epm': 'epm',
    'R2_valid_dirty': 'dirty'
}

comparacion['min_gap'] = comparacion[['gap', 'gap_epm', 'gap_dirty']].idxmin(axis=1).map(modelo_map)
comparacion['max_r2_train'] = comparacion[['R2_train', 'R2_train_epm', 'R2_train_dirty']].idxmax(axis=1).map(modelo_map)
comparacion['max_r2_valid'] = comparacion[['R2_valid', 'R2_valid_epm', 'R2_valid_dirty']].idxmax(axis=1).map(modelo_map)


In [145]:
comparacion

,region,R2_train,R2_valid,MAE_train,MAE_valid,RMSE_train,RMSE_valid,best_iterations,gap,R2_train_epm,R2_valid_epm,MAE_train_epm,MAE_valid_epm,RMSE_train_epm,RMSE_valid_epm,best_iterations_epm,gap_epm,R2_train_dirty,R2_valid_dirty,MAE_train_dirty,MAE_valid_dirty,RMSE_train_dirty,RMSE_valid_dirty,best_iterations_dirty,gap_dirty,min_gap,max_r2_train,max_r2_valid
0,18,0.999984,0.901764,0.007065,0.475877,6.880693e-05,0.539792,2991,0.098220,0.999984,0.901764,0.007065,0.475877,6.880693e-05,0.539792,2991,0.098220,0.999984,0.901764,0.007065,0.475877,6.880693e-05,0.539792,2991,0.098220,tp,tp,tp
1,5,0.999617,0.929657,0.040646,0.425339,2.371315e-03,0.504880,2997,0.069960,0.999617,0.929657,0.040646,0.425339,2.371315e-03,0.504880,2997,0.069960,0.999617,0.929657,0.040646,0.425339,2.371315e-03,0.504880,2997,0.069960,tp,tp,tp
2,4,0.999998,0.833281,0.003072,0.743596,1.307719e-05,1.283097,2196,0.166717,0.999998,0.833281,0.003072,0.743596,1.307719e-05,1.283097,2196,0.166717,0.999998,0.833281,0.003072,0.743596,1.307719e-05,1.283097,2196,0.166717,tp,tp,tp
3,10,0.998872,0.949704,0.079430,0.381049,9.270532e-03,0.404585,2993,0.049168,0.998872,0.949704,0.079430,0.381049,9.270532e-03,0.404585,2993,0.049168,0.998872,0.949704,0.079430,0.381049,9.270532e-03,0.404585,2993,0.049168,tp,tp,tp
4,22,1.000000,0.562962,0.000040,1.190931,1.913874e-09,3.673909,1652,0.437038,1.000000,0.562962,0.000040,1.190931,1.913874e-09,3.673909,1652,0.437038,1.000000,0.562962,0.000040,1.190931,1.913874e-09,3.673909,1652,0.437038,tp,tp,tp
5,9,0.989169,0.941368,0.145762,0.265904,3.568258e-02,0.189518,2985,0.047801,0.989169,0.941368,0.145762,0.265904,3.568258e-02,0.189518,2985,0.047801,0.989169,0.941368,0.145762,0.265904,3.568258e-02,0.189518,2985,0.047801,tp,tp,tp
6,21,0.999565,0.928841,0.045767,0.463063,3.003642e-03,0.525956,2999,0.070724,0.999565,0.928841,0.045767,0.463063,3.003642e-03,0.525956,2999,0.070724,0.999565,0.928841,0.045767,0.463063,3.003642e-03,0.525956,2999,0.070724,tp,tp,tp
7,20,1.000000,0.867077,0.000260,0.726611,8.974746e-08,0.842552,2980,0.132923,1.000000,0.867077,0.000260,0.726611,8.974746e-08,0.842552,2980,0.132923,1.000000,0.867077,0.000260,0.726611,8.974746e-08,0.842552,2980,0.132923,tp,tp,tp
8,12,0.997816,0.944985,0.088146,0.353910,1.179671e-02,0.311098,2995,0.052831,0.997816,0.944985,0.088146,0.353910,1.179671e-02,0.311098,2995,0.052831,0.997816,0.944985,0.088146,0.353910,1.179671e-02,0.311098,2995,0.052831,tp,tp,tp
9,8,0.999931,0.842503,0.014798,0.613522,2.899673e-04,0.686089,1482,0.157427,0.999931,0.842503,0.014798,0.613522,2.899673e-04,0.686089,1482,0.157427,0.999931,0.842503,0.014798,0.613522,2.899673e-04,0.686089,1482,0.157427,tp,tp,tp
